In [1]:
!pip install gym-super-mario-bros==7.3.2 -q
!pip install nes-py==8.2.1 -q
!pip install opencv-python -q

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import time

import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT, RIGHT_ONLY
from nes_py.wrappers import JoypadSpace

import cv2
import gym
from gym import spaces
from collections import deque
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None -- TURN ON GPU!'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.6/198.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.5 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using device: cuda
GPU: Tesla T4


In [2]:
import nes_py._rom as _rom_module
import gym_super_mario_bros.smb_env as _smb_env
import gym.wrappers.time_limit as _time_limit
import numpy as np

def _patched_prg_rom_stop(self):
    return int(self.prg_rom_start) + int(self.prg_rom_size) * 1024

def _patched_chr_rom_stop(self):
    return int(self.chr_rom_start) + int(self.chr_rom_size) * 1024

_rom_module.ROM.prg_rom_stop = property(_patched_prg_rom_stop)
_rom_module.ROM.chr_rom_stop = property(_patched_chr_rom_stop)

def _patched_x_position(self):
    return int(self.ram[0x6d]) * 0x100 + int(self.ram[0x86])

def _patched_y_pixel(self):
    return int(self.ram[0x03b8])

def _patched_y_position(self):
    return int(self.ram[0x00b5])

_smb_env.SuperMarioBrosEnv._x_position = property(_patched_x_position)
_smb_env.SuperMarioBrosEnv._y_pixel = property(_patched_y_pixel)
_smb_env.SuperMarioBrosEnv._y_position = property(_patched_y_position)

np.bool8 = np.bool_

def _patched_time_limit_step(self, action):
    observation, reward, done, info = self.env.step(action)
    self._elapsed_steps += 1
    if self._elapsed_steps >= self._max_episode_steps:
        info["TimeLimit.truncated"] = not done
        done = True
    return observation, reward, done, info

_time_limit.TimeLimit.step = _patched_time_limit_step

print("nes-py ROM patch applied")
print("smb_env NumPy 2.0 patch applied")
print("np.bool8 patch applied")
print("TimeLimit.step patch applied")

nes-py ROM patch applied
smb_env NumPy 2.0 patch applied
np.bool8 patch applied
TimeLimit.step patch applied


In [3]:
import subprocess as sp
import torch.multiprocessing as mp
def process_frame(frame):
    if frame is not None:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        frame = cv2.resize(frame, (84, 84))[None, :, :] / 255.
        return frame
    else:
        return np.zeros((1, 84, 84))


class CustomReward(gym.Wrapper):
    def __init__(self, env=None, world=None, stage=None):
        super(CustomReward, self).__init__(env)
        self.observation_space = spaces.Box(low=0, high=255, shape=(1, 84, 84))
        self.curr_score = 0
        self.current_x = 40
        self.world = world
        self.stage = stage

    def step(self, action):
        result = self.env.step(action)
        if len(result) == 5:
            state, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            state, reward, done, info = result

        state = process_frame(state)
        reward += (info["score"] - self.curr_score) / 40.
        self.curr_score = info["score"]
        if done:
            if info["flag_get"]:
                reward += 50
            else:
                reward -= 50

        if self.world == 4 and self.stage == 4:
            if (info["x_pos"] <= 1500 and info["y_pos"] < 127) or (
                    1588 <= info["x_pos"] < 2380 and info["y_pos"] >= 127):
                reward = -50
                done = True

        if self.world == 7 and self.stage == 4:
            if (506 <= info["x_pos"] <= 832 and info["y_pos"] > 127) or (
                    832 < info["x_pos"] <= 1064 and info["y_pos"] < 80) or (
                    1113 < info["x_pos"] <= 1464 and info["y_pos"] < 191) or (
                    1579 < info["x_pos"] <= 1943 and info["y_pos"] < 191) or (
                    1946 < info["x_pos"] <= 1964 and info["y_pos"] >= 191) or (
                    1984 < info["x_pos"] <= 2060 and (info["y_pos"] >= 191 or info["y_pos"] < 127)) or (
                    2114 < info["x_pos"] < 2440 and info["y_pos"] < 191) or info["x_pos"] < self.current_x - 500:
                reward -= 50
                done = True

        if self.world == 8 and self.stage == 4:
            if info["x_pos"] < self.current_x - 200:
                reward -= 50
                done = True

        self.current_x = info["x_pos"]
        return state, reward / 10., done, info

    def reset(self):
        self.curr_score = 0
        self.current_x = 40
        result = self.env.reset()
        if isinstance(result, tuple):
            state = result[0]
        else:
            state = result
        return process_frame(state)


class CustomSkipFrame(gym.Wrapper):
    def __init__(self, env, skip=4):
        super(CustomSkipFrame, self).__init__(env)
        self.observation_space = spaces.Box(low=0, high=255, shape=(skip, 84, 84))
        self.skip = skip
        self.states = np.zeros((skip, 84, 84), dtype=np.float32)

    def step(self, action):
        total_reward = 0
        last_states = []
        for i in range(self.skip):
            state, reward, done, info = self.env.step(action)
            total_reward += reward
            if i >= self.skip / 2:
                last_states.append(state)
            if done:
                self.reset()
                return self.states.astype(np.float32), total_reward, done, info
        max_state = np.max(np.concatenate(last_states, 0), 0)
        self.states[:-1] = self.states[1:]
        self.states[-1] = max_state
        return self.states.astype(np.float32), total_reward, done, info

    def reset(self):
        result = self.env.reset()
        if isinstance(result, tuple):
            state = result[0]
        else:
            state = result
        self.states = np.concatenate([state for _ in range(self.skip)], 0)
        return self.states.astype(np.float32)

def make_env(world, stage):
    env = gym_super_mario_bros.make(f"SuperMarioBros-{world}-{stage}-v0")
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    env = CustomReward(env, world, stage)
    env = CustomSkipFrame(env)
    return env

print("Environment wrappers ready")

Environment wrappers ready


In [4]:
class ActorCritic(nn.Module):
    def __init__(self, num_inputs, num_actions):
        super(ActorCritic, self).__init__()
        self.conv1 = nn.Conv2d(num_inputs, 32, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.conv4 = nn.Conv2d(32, 32, 3, stride=2, padding=1)
        self.fc = nn.Linear(32 * 6 * 6, 512)
        self.actor = nn.Linear(512, num_actions)
        self.critic = nn.Linear(512, 1)
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
                nn.init.orthogonal_(module.weight, nn.init.calculate_gain('relu'))
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        x = x.float() / 255.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.fc(x.view(x.size(0), -1)))
        return x

    def get_action(self, x):
        x = self.forward(x)
        policy = F.softmax(self.actor(x), dim=-1)
        dist = Categorical(policy)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob

    def evaluate(self, x, action):
        x = self.forward(x)
        policy = F.softmax(self.actor(x), dim=-1)
        dist = Categorical(policy)
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        value = self.critic(x)
        return log_prob, entropy, value


num_actions = len(SIMPLE_MOVEMENT)
model = ActorCritic(4, num_actions).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Actions: {num_actions}")
print(f"Action space: {SIMPLE_MOVEMENT}")

Model parameters: 623,368
Actions: 7
Action space: [['NOOP'], ['right'], ['right', 'A'], ['right', 'B'], ['right', 'A', 'B'], ['A'], ['left']]


In [5]:
class PPOAgent:
    def __init__(self, model, lr=1e-4, gamma=0.9, lam=1.0,
                 clip_epsilon=0.2, epochs=10, batch_size=256, beta=0.01):
        self.model = model
        self.optimizer = optim.Adam(model.parameters(), lr=lr)
        self.gamma = gamma
        self.lam = lam
        self.clip_epsilon = clip_epsilon
        self.epochs = epochs
        self.batch_size = batch_size
        self.beta = beta

    def compute_gae(self, rewards, values, dones, next_value):
        advantages = []
        gae = 0
        values = values + [next_value]
        for step in reversed(range(len(rewards))):
            delta = rewards[step] + self.gamma * values[step + 1] * (1 - dones[step]) - values[step]
            gae = delta + self.gamma * self.lam * (1 - dones[step]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, values[:-1])]
        return advantages, returns

    def learn(self, states, actions, old_log_probs, advantages, returns):
        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.LongTensor(actions).to(device)
        old_log_probs = torch.FloatTensor(old_log_probs).to(device)
        advantages = torch.FloatTensor(advantages).to(device)
        returns = torch.FloatTensor(returns).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        for _ in range(self.epochs):
            indices = torch.randperm(len(states))
            for start in range(0, len(states), self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                batch_states = states[batch_idx]
                batch_actions = actions[batch_idx]
                batch_old_log_probs = old_log_probs[batch_idx]
                batch_advantages = advantages[batch_idx]
                batch_returns = returns[batch_idx]

                new_log_probs, entropy, values_pred = self.model.evaluate(batch_states, batch_actions)
                ratio = torch.exp(new_log_probs - batch_old_log_probs)
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * batch_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = F.smooth_l1_loss(batch_returns, values_pred.squeeze())
                entropy_loss = entropy.mean()
                loss = actor_loss + critic_loss - self.beta * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                self.optimizer.step()

    def collect_and_learn(self, env, num_steps=512):
        states, actions, rewards, dones, log_probs, values = [], [], [], [], [], []
        state = env.reset()
        episode_reward = 0
        total_reward = 0
        episodes = 0

        for _ in range(num_steps):
            state_tensor = torch.FloatTensor(np.array(state)).unsqueeze(0).to(device)
            with torch.no_grad():
                action, log_prob = self.model.get_action(state_tensor)
                _, _, value = self.model.evaluate(state_tensor, torch.LongTensor([action]).to(device))

            next_state, reward, done, info = env.step(action)
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            dones.append(float(done))
            log_probs.append(log_prob.item())
            values.append(value.item())
            episode_reward += reward
            state = next_state

            if done:
                total_reward += episode_reward
                episodes += 1
                episode_reward = 0
                state = env.reset()

        next_state_tensor = torch.FloatTensor(np.array(state)).unsqueeze(0).to(device)
        with torch.no_grad():
            _, _, next_value = self.model.evaluate(next_state_tensor, torch.LongTensor([0]).to(device))

        advantages, returns = self.compute_gae(rewards, values, dones, next_value.item())
        self.learn(states, actions, log_probs, advantages, returns)
        avg_reward = total_reward / max(episodes, 1)
        return avg_reward, episodes


agent = PPOAgent(model)
print("PPO Agent ready")

PPO Agent ready


In [6]:
def train(world, stage, num_updates=10000, save_interval=50, checkpoint_path="/kaggle/working/mario_ppo.pt"):
    env = make_env(world, stage)
    num_actions = len(SIMPLE_MOVEMENT)
    model = ActorCritic(4, num_actions).to(device)
    agent = PPOAgent(model)
    start_update = 0
    rewards_history = []

    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model"])
        agent.optimizer.load_state_dict(checkpoint["optimizer"])
        start_update = checkpoint["update"]
        rewards_history = checkpoint.get("rewards_history", [])
        print(f"Resumed from update {start_update}")
    else:
        print("Starting fresh training")

    print(f"Training on World {world}-{stage} | Device: {device}")
    print(f"Updates: {num_updates} | Save every: {save_interval} updates")
    print("-" * 50)

    start_time = time.time()

    for update in range(start_update, num_updates):
        avg_reward, episodes = agent.collect_and_learn(env, num_steps=512)
        rewards_history.append(avg_reward)

        if update % 10 == 0:
            elapsed = time.time() - start_time
            recent_avg = np.mean(rewards_history[-10:]) if len(rewards_history) >= 10 else np.mean(rewards_history)
            print(f"Update {update}/{num_updates} | Reward: {avg_reward:.2f} | "
                  f"10-avg: {recent_avg:.2f} | Episodes: {episodes} | Time: {elapsed:.0f}s")

        if update % save_interval == 0 and update > 0:
            torch.save({
                "model": model.state_dict(),
                "optimizer": agent.optimizer.state_dict(),
                "update": update,
                "rewards_history": rewards_history
            }, checkpoint_path)
            print(f"Checkpoint saved at update {update}")

    env.close()
    torch.save({
        "model": model.state_dict(),
        "optimizer": agent.optimizer.state_dict(),
        "update": num_updates,
        "rewards_history": rewards_history
    }, checkpoint_path)
    print("Training complete. Final model saved.")
    return model, rewards_history

In [7]:
def sanity_check():
    print("Running sanity check...")
    env = make_env(1, 1)
    obs = env.reset()
    print(f"Observation shape: {obs.shape}")

    state_tensor = torch.FloatTensor(np.array(obs)).unsqueeze(0).to(device)
    print(f"State tensor shape: {state_tensor.shape}")

    test_model = ActorCritic(4, len(SIMPLE_MOVEMENT)).to(device)
    test_agent = PPOAgent(test_model)

    action, log_prob = test_model.get_action(state_tensor)
    print(f"Action: {action} ({SIMPLE_MOVEMENT[action]})")
    print(f"Log prob: {log_prob.item():.4f}")

    _, _, value = test_model.evaluate(state_tensor, torch.LongTensor([action]).to(device))
    print(f"Value estimate: {value.item():.4f}")

    print("\nRunning 10 random steps...")
    total_reward = 0
    for i in range(10):
        action, _ = test_model.get_action(state_tensor)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        state_tensor = torch.FloatTensor(np.array(obs)).unsqueeze(0).to(device)
        if done:
            obs = env.reset()
            state_tensor = torch.FloatTensor(np.array(obs)).unsqueeze(0).to(device)

    print(f"10 steps completed | Total reward: {total_reward:.4f}")
    print(f"Mario x position: {info['x_pos']}")
    print(f"Mario y position: {info['y_pos']}")
    print(f"Flag get: {info['flag_get']}")

    print("\nTesting mini PPO update (50 steps)...")
    avg_reward, episodes = test_agent.collect_and_learn(env, num_steps=50)
    print(f"Mini update complete | Avg reward: {avg_reward:.4f}")

    env.close()
    print("\nAll checks passed. Ready to train!")

sanity_check()

Running sanity check...


/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-1-1-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(


Observation shape: (4, 84, 84)
State tensor shape: torch.Size([1, 4, 84, 84])
Action: 3 (['right', 'B'])
Log prob: -1.9444
Value estimate: -0.0014

Running 10 random steps...


/usr/local/lib/python3.12/dist-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(


10 steps completed | Total reward: 2.0000
Mario x position: 62
Mario y position: 1
Flag get: False

Testing mini PPO update (50 steps)...
Mini update complete | Avg reward: 0.0000

All checks passed. Ready to train!


In [8]:
import time
# Train on World 1-1 for 5000 updates (adjust as needed)
model, history = train(world=1, stage=1, num_updates=5000, save_interval=500)

Starting fresh training
Training on World 1-1 | Device: cuda
Updates: 5000 | Save every: 500 updates
--------------------------------------------------
Update 0/5000 | Reward: 58.40 | 10-avg: 58.40 | Episodes: 1 | Time: 9s
Update 10/5000 | Reward: 69.23 | 10-avg: 57.49 | Episodes: 4 | Time: 96s
Update 20/5000 | Reward: 62.53 | 10-avg: 65.93 | Episodes: 3 | Time: 180s
Update 30/5000 | Reward: 59.85 | 10-avg: 55.02 | Episodes: 5 | Time: 262s
Update 40/5000 | Reward: 81.25 | 10-avg: 78.64 | Episodes: 3 | Time: 345s
Update 50/5000 | Reward: 72.80 | 10-avg: 71.50 | Episodes: 4 | Time: 427s
Update 60/5000 | Reward: 56.23 | 10-avg: 61.95 | Episodes: 6 | Time: 510s
Update 70/5000 | Reward: 54.04 | 10-avg: 70.75 | Episodes: 6 | Time: 591s
Update 80/5000 | Reward: 74.17 | 10-avg: 58.56 | Episodes: 3 | Time: 667s
Update 90/5000 | Reward: 65.15 | 10-avg: 62.31 | Episodes: 4 | Time: 744s
Update 100/5000 | Reward: 78.75 | 10-avg: 75.00 | Episodes: 2 | Time: 820s
Update 110/5000 | Reward: 63.55 | 10-